# External Data fusion
`Build the contextual environment dataset completely isolated from the modeling logic`

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
BASE_DIR = Path().resolve().parent

In [12]:

#* time series dataset (to extract timestamps)
time_series_path = BASE_DIR / 'Dataset' / 'ai4i2020_time_series.csv'
internal_df = pd.read_csv(time_series_path)

timestamps = pd.to_datetime(internal_df['Timestamp']).unique()
total_rows = len(timestamps)
print(f'extracted temporal anchor of {total_rows} minute-by-minute operational cycles')

extracted temporal anchor of 10000 minute-by-minute operational cycles


In [13]:

#? extract time components
hours = pd.DatetimeIndex(timestamps).hour
days = pd.DatetimeIndex(timestamps).day

#todo set random seed to prevent 'random randomness'!
np.random.seed(42)

## External feature generation
`8 contextual environment signals derived purely from the extracted hours / days / total_rows anchors`

In [14]:
#* External feature generation — 8 contextual environment signals derived from hours / days / total_rows

# 1. ambient_temp (Ambient Temperature, Kelvin)
ambient_temp = 298.15 + 6.0 * np.cos(2 * np.pi * (hours - 14) / 24) + np.random.normal(0, 0.4, total_rows)

# 2. ambient_humidity (Ambient Humidity, %)
ambient_humidity = 60.0 - 12.0 * np.cos(2 * np.pi * (hours - 14) / 24) + np.random.normal(0, 1.0, total_rows)

# 3. atmospheric_pressure (hPa)
atmospheric_pressure = 1013.25 + 3.0 * np.sin(2 * np.pi * days / 5) + np.random.normal(0, 0.5, total_rows)

# 4. grid_voltage_fluctuation (V)
grid_voltage_fluctuation = np.random.normal(0, 1.5, total_rows) + np.where(
    np.random.rand(total_rows) > 0.98,
    np.random.uniform(-5.0, -2.0, total_rows),
    0
)

# 5. factory_load_density (Floor Congestion, 0-1)
factory_load_density = np.where(
    (hours >= 8) & (hours <= 20),
    np.random.uniform(0.75, 0.95, total_rows),
    np.random.uniform(0.30, 0.55, total_rows)
)

# 6. operator_skill_proxy (Shift Experience Level)
operator_skill_proxy = np.where(
    (hours >= 8) & (hours < 16), 1,
    np.where((hours >= 16) & (hours < 24), 2, 3)
)

# 7. particulate_matter_pm10 (Airborne Dust)
particulate_matter_pm10 = 35.0 + (factory_load_density * 25.0) + np.random.exponential(5.0, total_rows)

# 8. ambient_vibration_noise (Background Decibels)
ambient_vibration_noise = 45.0 + (factory_load_density * 15.0) + np.random.normal(0, 1.2, total_rows)

### Assemble the external environment dataset

In [15]:
#Created a Dataframe for the features
external_env_df = pd.DataFrame({
    'Timestamp': timestamps,
    'ambient_temp': ambient_temp,
    'ambient_humidity': ambient_humidity,
    'atmospheric_pressure': atmospheric_pressure,
    'grid_voltage_fluctuation': grid_voltage_fluctuation,
    'factory_load_density': factory_load_density,
    'operator_skill_proxy': operator_skill_proxy,
    'particulate_matter_pm10': particulate_matter_pm10,
    'ambient_vibration_noise': ambient_vibration_noise,
})

external_env_df.head()

,Timestamp,ambient_temp,ambient_humidity,atmospheric_pressure,grid_voltage_fluctuation,factory_load_density,operator_skill_proxy,particulate_matter_pm10,ambient_vibration_noise
0,2026-01-01 00:00:00,293.152533,69.713810,1016.277313,-2.970858,0.349519,3,47.204333,50.949400
1,2026-01-01 00:01:00,292.898542,70.086805,1016.244831,-1.582478,0.343729,3,57.006136,48.118674
2,2026-01-01 00:02:00,293.212923,69.794924,1015.634910,-0.880543,0.372390,3,57.624113,50.540886
3,2026-01-01 00:03:00,293.563060,70.502723,1016.392962,0.224503,0.325542,3,45.330149,50.883647
4,2026-01-01 00:04:00,292.860186,71.589483,1015.358128,1.536243,0.313322,3,46.462858,51.716968


In [16]:
for var in globals():
    if 'df' in var.lower():
        print(var)

RuntimeError: dictionary changed size during iteration

In [17]:
fused_df = pd.concat([internal_df, external_env_df], axis=1)

In [18]:
# 2. ADVANCED CROSS-DOMAIN BRAINS (The Interaction Layer)
print("Engineering advanced physics-based interaction features...")

# Baseline Features
fused_df['Thermal_Strain'] = (
    fused_df['Process temperature [K]'] - fused_df['ambient_temp']
)

fused_df['Total_Stress_Index'] = (
    fused_df['Torque [Nm]'] * fused_df['factory_load_density']
)

# A. Heat Dissipation Efficiency Ratio
fused_df['Thermal_Gradient_Ratio'] = (
    fused_df['Air temperature [K]'] / fused_df['ambient_temp']
)

# B. Mechanical Resonance Stress
fused_df['Vibration_Torque_Impact'] = (
    fused_df['Torque [Nm]'] * fused_df['ambient_vibration_noise']
)

# C. Electrical/Thermal Combined Severity
fused_df['Electrical_Thermal_Stress'] = (
    np.abs(fused_df['grid_voltage_fluctuation']) *
    fused_df['Process temperature [K]']
)

# 3. EXPANDED MULTI-WINDOW ROLLING CONTEXT
print("Processing short-term and long-term rolling signals...")

windows = [5, 30]

for w in windows:

    # Air Temperature Variance & Mean
    fused_df[f'Air_temp_var_{w}'] = (
        fused_df['Air temperature [K]']
        .rolling(window=w)
        .var()
        .bfill()
    )

    fused_df[f'Air_temp_mean_{w}'] = (
        fused_df['Air temperature [K]']
        .rolling(window=w)
        .mean()
        .bfill()
    )

    # Process Temperature Variance & Mean
    fused_df[f'Process_temp_var_{w}'] = (
        fused_df['Process temperature [K]']
        .rolling(window=w)
        .var()
        .bfill()
    )

    fused_df[f'Process_temp_mean_{w}'] = (
        fused_df['Process temperature [K]']
        .rolling(window=w)
        .mean()
        .bfill()
    )

    # Dust Exposure Trend
    fused_df[f'Dust_Exposure_mean_{w}'] = (
        fused_df['particulate_matter_pm10']
        .rolling(window=w)
        .mean()
        .bfill()
    )

Engineering advanced physics-based interaction features...
Processing short-term and long-term rolling signals...


C:\Users\anasq\AppData\Local\Temp\ipykernel_15756\1923317560.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fused_df['Thermal_Strain'] = (
C:\Users\anasq\AppData\Local\Temp\ipykernel_15756\1923317560.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fused_df['Total_Stress_Index'] = (
C:\Users\anasq\AppData\Local\Temp\ipykernel_15756\1923317560.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all column

In [19]:
fused_df.shape

(10000, 149)

In [20]:

#? export as csv file
fused_output_path = BASE_DIR / 'Dataset' / 'predictive_maintenance_master_features.csv'
fused_df.to_csv(fused_output_path, index=False)